In [1]:
import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
df = pd.read_csv(url)


df = df.drop(['UDI', 'Product ID'], axis=1)


display(df.head())


print("\n--- Target Variable Distribution (0: Operational, 1: Failed) ---")
print(df['Machine failure'].value_counts())

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0



--- Target Variable Distribution (0: Operational, 1: Failed) ---
Machine failure
0    9661
1     339
Name: count, dtype: int64


In [2]:
quality_mapping = {'L': 0, 'M': 1, 'H': 2}
df['Type'] = df['Type'].map(quality_mapping)

leakage_columns = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
df = df.drop(columns=leakage_columns)

X = df.drop('Machine failure', axis=1)
y = df['Machine failure']

display(X.head())

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
0,1,298.1,308.6,1551,42.8,0
1,0,298.2,308.7,1408,46.3,3
2,0,298.1,308.5,1498,49.4,5
3,0,298.2,308.6,1433,39.5,7
4,0,298.2,308.7,1408,40.0,9


In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

X.columns = X.columns.str.replace('[', '', regex=False).str.replace(']', '', regex=False).str.replace('<', '', regex=False)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

imbalance_ratio = y_train.value_counts()[0] / y_train.value_counts()[1]

xgb_model = XGBClassifier(
    scale_pos_weight=imbalance_ratio,
    n_estimators=100,
    max_depth=5,
    random_state=42,
    eval_metric='auc'
)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)

print("--- Kurumsal Model Başarı Raporu (Classification Report) ---")
print(classification_report(y_test, y_pred))

print("\n--- Hata Matrisi (Confusion Matrix) ---")
print(confusion_matrix(y_test, y_pred))

ValueError: feature_names must be string, and may not contain [, ] or <